# Debug Drill 16: CNN Overfitting

## The Situation

Your colleague built a CNN to classify Fashion-MNIST images. They're excited because training accuracy reached 99%! But when deployed, the model performs poorly on new images.

## The Symptom

- Training accuracy: ~99%
- Test accuracy: ~65-70%
- Users complain the model misclassifies obvious items

## Your Task

1. **Identify** what's wrong
2. **Fix** the model
3. **Verify** the fix works
4. **Write** a 3-bullet postmortem

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
# Load data (using subset to exaggerate overfitting)
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# Use small subset for training (makes overfitting worse)
X_train = X_train_full[:2000].astype('float32') / 255.0
y_train = y_train_full[:2000]
X_test = X_test.astype('float32') / 255.0

# Reshape for CNN
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## The Buggy Model

Your colleague's CNN that achieves 99% training accuracy:

In [ ]:
# ========================================
# BUGGY CODE - DO NOT MODIFY (yet)
# ========================================

buggy_model = keras.Sequential([
    # Very deep architecture for small dataset
    layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(256, (3, 3), activation='relu'),
    
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

buggy_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Model parameters: {buggy_model.count_params():,}")

In [ ]:
# Train the buggy model
print("Training buggy model...")
history = buggy_model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluate
train_loss, train_acc = buggy_model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = buggy_model.evaluate(X_test, y_test, verbose=0)

print(f"\n=== Results ===")
print(f"Training Accuracy: {train_acc:.1%}")
print(f"Test Accuracy: {test_acc:.1%}")
print(f"Gap: {(train_acc - test_acc)*100:.1f} percentage points")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Over Time')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Over Time')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Your Investigation

### Step 1: Identify the Bug

Look at:
- The training curves above
- The gap between train and test accuracy
- The model architecture
- The amount of training data

**What's wrong?**

In [ ]:
# YOUR ANALYSIS HERE
# Examine the training curves, model size, data size, etc.

print("=== Diagnostic Checks ===")
print(f"Training samples: {len(X_train)}")
print(f"Model parameters: {buggy_model.count_params():,}")
print(f"Ratio (params/samples): {buggy_model.count_params() / len(X_train):.1f}")
print(f"Train-test gap: {(train_acc - test_acc)*100:.1f}%")

# TODO: Write your diagnosis
# The bug is: ...

### Step 2: Fix the Model

Create a fixed version that doesn't overfit. Consider:
- Adding Dropout layers
- Adding BatchNormalization
- Using a smaller/simpler architecture
- Using data augmentation
- Early stopping

In [ ]:
# TODO: Build your fixed model

fixed_model = keras.Sequential([
    # YOUR FIXED ARCHITECTURE HERE
    # Hint: Add Dropout and BatchNormalization
    # Hint: Consider a simpler architecture
    
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    # ... add your layers ...
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

fixed_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Fixed model parameters: {fixed_model.count_params():,}")

In [ ]:
# TODO: Train your fixed model with early stopping

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("Training fixed model...")
fixed_history = fixed_model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

### Step 3: Verify the Fix

In [ ]:
# Evaluate fixed model
fixed_train_loss, fixed_train_acc = fixed_model.evaluate(X_train, y_train, verbose=0)
fixed_test_loss, fixed_test_acc = fixed_model.evaluate(X_test, y_test, verbose=0)

print("\n=== Comparison ===")
print(f"{'Metric':<20} {'Buggy':<15} {'Fixed':<15}")
print("-" * 50)
print(f"{'Train Accuracy':<20} {train_acc:<15.1%} {fixed_train_acc:<15.1%}")
print(f"{'Test Accuracy':<20} {test_acc:<15.1%} {fixed_test_acc:<15.1%}")
print(f"{'Gap':<20} {(train_acc-test_acc)*100:<15.1f}% {(fixed_train_acc-fixed_test_acc)*100:<15.1f}%")
print(f"{'Parameters':<20} {buggy_model.count_params():<15,} {fixed_model.count_params():<15,}")

In [ ]:
# Verify improvement
buggy_gap = train_acc - test_acc
fixed_gap = fixed_train_acc - fixed_test_acc

if fixed_gap < buggy_gap * 0.5:  # Gap reduced by at least 50%
    print("\n✅ Fix verified! Overfitting significantly reduced.")
else:
    print("\n⚠️ Gap still too large. Try more regularization.")

### Step 4: Write Your Postmortem

Write 3 bullets explaining:
1. What was wrong
2. How you fixed it
3. How to prevent this in the future

## Your Postmortem

- **Root cause:** ...
- **Fix applied:** ...
- **Prevention:** ...

---